In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from restaurant_visitor_eda.config import PROCESSED_DATA_DIR

df_train_raw = pd.read_csv(PROCESSED_DATA_DIR / "train_features.csv")
df_test_raw = pd.read_csv(PROCESSED_DATA_DIR / "test_features.csv")

In [ ]:
categorical_features = [
    "air_store_id",
    "air_genre_name",
    "day_of_week",
    "month",
    "day_pattern",
    "prefecture",
    "district",
    "block",
]

numeric_features = [
    "latitude",
    "longitude",
    "doy_sin",
    "doy_cos",
    "dow_sin",
    "dow_cos",
    "median_visitors_dow",
    "mean_visitors_dow",
    "min_visitors_dow",
    "max_visitors_dow",
    "median_visitors_total",
    "gw_genre_geo_median",
    "median_reserve_visitors_dow",
    "walk_in_ratio",
]

binary_features = [
    "is_gw",
    "has_gw_history",
]

features = categorical_features + numeric_features + binary_features

df_train_raw[categorical_features] = (
    df_train_raw[categorical_features].fillna("Unknown").astype(str)
)
df_test_raw[categorical_features] = df_test_raw[categorical_features].fillna("Unknown").astype(str)

X_train = df_train_raw[features]
y_train = np.log1p(df_train_raw["visitors"])

X_test = df_test_raw[features]

In [ ]:
from catboost import CatBoostRegressor
import numpy as np
import pandas as pd

model = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    eval_metric="RMSE",
    cat_features=categorical_features,
    random_seed=42,
    verbose=100,
)

In [ ]:
model.fit(X_train, y_train)

0:	learn: 0.7855920	total: 96.6ms	remaining: 2m 24s
100:	learn: 0.5087339	total: 8.53s	remaining: 1m 58s
200:	learn: 0.5003580	total: 16.8s	remaining: 1m 48s
300:	learn: 0.4956977	total: 25.6s	remaining: 1m 42s
400:	learn: 0.4923758	total: 34.2s	remaining: 1m 33s
500:	learn: 0.4901352	total: 42.8s	remaining: 1m 25s
600:	learn: 0.4882431	total: 51.5s	remaining: 1m 17s
700:	learn: 0.4865852	total: 1m	remaining: 1m 8s
800:	learn: 0.4849861	total: 1m 9s	remaining: 1m
900:	learn: 0.4834699	total: 1m 17s	remaining: 51.8s
1000:	learn: 0.4820694	total: 1m 26s	remaining: 43.2s
1100:	learn: 0.4807059	total: 1m 35s	remaining: 34.7s
1200:	learn: 0.4795118	total: 1m 44s	remaining: 26.1s
1300:	learn: 0.4783479	total: 1m 53s	remaining: 17.4s
1400:	learn: 0.4770322	total: 2m 3s	remaining: 8.7s
1499:	learn: 0.4759339	total: 2m 12s	remaining: 0us


CatBoostRegressor(cat_features=['air_store_id', 'air_genre_name', 'day_of_week', 'month', 'day_pattern', 'prefecture', 'district', 'block'], depth=8, eval_metric='RMSE', iterations=1500, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

In [ ]:
preds_log = model.predict(X_test)

preds_real = np.expm1(preds_log)


df_test_raw["visit_date"] = pd.to_datetime(df_test_raw["visit_date"])

submission = pd.DataFrame(
    {
        "id": df_test_raw["air_store_id"]
        + "_"
        + df_test_raw["visit_date"].dt.strftime("%Y-%m-%d"),
        "visitors": preds_real,
    }
)

submission_path = "submission_catboost_v1.csv"
submission.to_csv(submission_path, index=False)

submission.head()

,id,visitors
0,air_00a91d42b08b08d9_2017-04-23,2.095474
1,air_08cb3c4ee6cd6a22_2017-04-23,15.323213
2,air_f8233ad00755c35c_2017-04-23,5.605892
3,air_234d3dbf7f3d5a50_2017-04-23,8.917495
4,air_a563896da3777078_2017-04-23,33.589448
